In [40]:
import os
import re
import time
import pathlib
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import json

In [41]:
csv_path = "E:/2026_capstone/policy_data/hud.csv"
output_path = "E:/2026_capstone/policy_data/hud_data"

RAW_DIR  = os.path.join(output_path, "raw_text")
XML_DIR  = os.path.join(output_path, "xml")
META_DIR = os.path.join(output_path, "meta")

for d in [RAW_DIR, XML_DIR, META_DIR]:
    os.makedirs(d, exist_ok=True)

In [42]:
def safe_filename(s: str) -> str:
    s = str(s)
    s = re.sub(r"[^\w\-\.]+", "_", s)
    return s[:180]

def build_session():
    sess = requests.Session()
    retry = Retry(
        total=5,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )  
    adapter = HTTPAdapter(max_retries=retry)
    sess.mount("http://", adapter)
    sess.mount("https://", adapter)
    sess.headers.update({
        "User-Agent": "CUSP-RAG-Downloader/1.0 (research use; polite rate limiting)",
        "Accept": "application/json,text/plain,*/*"
    })
    return sess


In [43]:
def http_get_json(session: requests.Session, url: str) -> dict:
    r = session.get(url, timeout=60)
    r.raise_for_status()
    return r.json()

def download_text(session: requests.Session, url: str, out_path: str):
    if not url:
        return False, "empty_url"

    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        return True, "exists"

    try:
        r = session.get(url, timeout=120)
        r.raise_for_status()
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(r.text)
        return True, "downloaded"
    except Exception as e:
        return False, str(e)

In [44]:
def main():
    df = pd.read_csv(csv_path, encoding="utf-8", encoding_errors="replace")
    session = build_session()

    for i, row in df.iterrows():
        docno = str(row.get("document_number", "")).strip()
        pub   = str(row.get("publication_date", "")).strip()

        if not docno:
            continue

        api_url = f"https://www.federalregister.gov/api/v1/documents/{docno}.json"
        try:
            doc = http_get_json(session, api_url)
        except Exception as e:
            print(f"[{i+1}/{len(df)}] API failed {docno}: {e}")
            time.sleep(0.8)
            continue

        raw_text_url = doc.get("raw_text_url")
        xml_url      = doc.get("full_text_xml_url")

        base = safe_filename(f"{pub}_{docno}")
        raw_path  = os.path.join(RAW_DIR,  base + ".txt")
        xml_path  = os.path.join(XML_DIR,  base + ".xml")
        meta_path = os.path.join(META_DIR, base + ".json")

        ok_raw, msg_raw = download_text(session, raw_text_url, raw_path)
        ok_xml, msg_xml = download_text(session, xml_url,      xml_path)

        meta = {
            "document_number": docno,
            "publication_date": doc.get("publication_date") or pub,
            "title": doc.get("title"),
            "type": doc.get("type"),
            "agency_names": doc.get("agency_names"),
            "citation": doc.get("citation"),
            "html_url": doc.get("html_url"),
            "json_url": doc.get("json_url"),
            "raw_text_url": raw_text_url,
            "full_text_xml_url": xml_url,
            "raw_saved": ok_raw,
            "xml_saved": ok_xml,
            "raw_status": msg_raw,
            "xml_status": msg_xml,
        }

        pathlib.Path(meta_path).write_text(
            json.dumps(meta, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )

        time.sleep(0.35)

        if (i + 1) % 20 == 0:
            print(f"Processed {i+1}/{len(df)}")

    print("Done.")

if __name__ == "__main__":
    main()

Processed 20/34
Done.
